# Steganography: Watermarking Exercise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Iliad_steganography_and_backdoors/blob/main/exercises/watermarking/notebook.ipynb)

In [ ]:
import os, importlib
if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Iliad_steganography_and_backdoors/main/exercises/watermarking/utils.py",
        "utils.py"
    )
    importlib.invalidate_caches()

import utils
importlib.reload(utils)
from utils import (
    generate,
    test_prf, test_gumbel_sample, test_watermark_score, test_p_value, test_decode_message,
)

In [ ]:
import hashlib
import hmac

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
VOCAB = model.config.vocab_size
CONTEXT = 4
KEY = b"iliad"

## Overview

A language model writes text by sampling one token at a time from a distribution $p$ over the vocabulary. Sampling needs randomness, and normally nobody cares where that randomness comes from.

A watermark replaces the random numbers with **pseudorandom** numbers computed from a secret key. Every single token is still a fair sample from $p$, so the text looks like ordinary model output. But whoever holds the key can recompute the pseudorandom numbers and see that the text fits them suspiciously well.

You will build the scheme proposed by Scott Aaronson in 2022:

1. **Part A**: a pseudorandom function that turns (key, previous tokens) into random-looking numbers
2. **Part B**: a sampling rule that uses these numbers and still samples exactly from $p$
3. **Part C and D**: a detector that needs only the key and the text, not the model
4. **Experiments**: when does detection work, and when does it fail?
5. **Bonus**: hide an arbitrary message instead of a single bit, which turns the watermark into steganography

---

## Part A: A Pseudorandom Function

We need a function $F_k$ that maps the previous `CONTEXT` tokens to one number in $[0, 1)$ for **every** token in the vocabulary:

$$r = F_k(\text{context}) \in [0, 1)^{|V|}$$

Without the key $k$ the output should be indistinguishable from true randomness. With the key, anyone can recompute it.

We build it in two steps:
1. `HMAC-SHA256(key, context)` gives 32 bytes that look random to anyone without the key
2. These bytes seed a random number generator, which expands them into `vocab_size` uniform numbers

In [ ]:
def prf(key, context, vocab_size):
    message = np.array(context, dtype=np.int64).tobytes()
    # TODO (3 lines): compute the HMAC-SHA256 digest of message under key,
    # turn the digest into an integer seed,
    # and return vocab_size uniform numbers from np.random.default_rng(seed)
    pass

In [ ]:
test_prf(prf)

<details>
<summary><b>Hint</b></summary>

`hmac.new(key, message, hashlib.sha256).digest()` returns the bytes. `int.from_bytes(digest, "big")` turns them into an integer.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def prf(key, context, vocab_size):
    message = np.array(context, dtype=np.int64).tobytes()
    digest = hmac.new(key, message, hashlib.sha256).digest()
    seed = int.from_bytes(digest, "big")
    return np.random.default_rng(seed).random(vocab_size)
```
</details>

## Part B: Sampling with the Gumbel Trick

Given the model's probabilities $p$ and the pseudorandom numbers $r$, we output the token

$$x = \arg\max_v \; r_v^{1/p_v}$$

If $r$ is uniformly random, token $v$ wins with probability exactly $p_v$. This is the Gumbel-max trick, and it is why the watermark does not change the distribution of the text.

At the same time the winning token tends to have $r_v$ close to 1. The detector will look for exactly this.

Compute the argmax in log space, $\log(r_v) / p_v$, since $r_v^{1/p_v}$ underflows for small $p_v$.

In [ ]:
def gumbel_sample(probs, r):
    # TODO (1 line): return the index of the token maximising log(r) / probs
    pass

In [ ]:
test_gumbel_sample(gumbel_sample)

<details>
<summary><b>Hint</b></summary>

Use `np.argmax` and `np.log`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def gumbel_sample(probs, r):
    return int(np.argmax(np.log(r) / probs))
```
</details>

## Generating Text

`generate` from `utils.py` runs GPT-2 token by token. At every step it calls `sampler(probs, history, i)`, where `probs` is the next-token distribution, `history` holds all tokens so far (prompt included) and `i` counts the generated tokens.

The two samplers below differ only in where the randomness comes from.

In [ ]:
def plain_sampler(seed):
    rng = np.random.default_rng(seed)
    return lambda probs, history, i: rng.choice(len(probs), p=probs)


def watermark_sampler(key):
    return lambda probs, history, i: gumbel_sample(probs, prf(key, history[-CONTEXT:], len(probs)))


prompt = "The history of cryptography begins with"
plain = generate(model, tokenizer, prompt, plain_sampler(0), 100)
marked = generate(model, tokenizer, prompt, watermark_sampler(KEY), 100)

print("PLAIN:", tokenizer.decode(plain))
print()
print("WATERMARKED:", tokenizer.decode(marked))

Can you tell which one carries the watermark? Neither can anyone else without the key.

## Part C: The Detection Score

The detector knows the key and sees only the text. For every token $x_i$ it recomputes $r = F_k(x_{i-c}, \dots, x_{i-1})$ and looks up the entry $r_{x_i}$ of the token that was actually written. The score is

$$S = \sum_i -\ln(1 - r_{x_i})$$

Two details:
- The first `context_size` tokens are skipped, because their context includes the prompt, which the detector does not know
- Every (context, token) pair is counted only once. Repeated phrases would otherwise count the same evidence many times

Return the score and the number $n$ of pairs that were counted.

In [ ]:
def watermark_score(tokens, key, vocab_size, context_size=CONTEXT):
    seen = set()
    score = 0.0
    for i in range(context_size, len(tokens)):
        pair = (tuple(tokens[i - context_size:i]), tokens[i])
        # TODO (4 lines): skip pairs that were seen before, remember new ones,
        # and add -log(1 - r) to the score, where r belongs to the observed token
        pass
    return score, len(seen)

In [ ]:
test_watermark_score(watermark_score, prf)

<details>
<summary><b>Hint</b></summary>

`prf(key, pair[0], vocab_size)[pair[1]]` is the pseudorandom number of the observed token.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def watermark_score(tokens, key, vocab_size, context_size=CONTEXT):
    seen = set()
    score = 0.0
    for i in range(context_size, len(tokens)):
        pair = (tuple(tokens[i - context_size:i]), tokens[i])
        if pair in seen:
            continue
        seen.add(pair)
        score += -np.log(1 - prf(key, pair[0], vocab_size)[pair[1]])
    return score, len(seen)
```
</details>

## Part D: From Score to p-Value

How large is a suspicious score? Consider text that was **not** watermarked with our key. Then each $r_{x_i}$ is just a uniform random number, so $-\ln(1 - r_{x_i})$ follows an exponential distribution with mean 1. The sum of $n$ of them follows a Gamma distribution with shape $n$ and scale 1.

The p-value is the probability that unwatermarked text reaches a score of at least $S$ by chance:

$$\text{p-value} = \Pr[\text{Gamma}(n, 1) \geq S]$$

In [ ]:
def p_value(score, n):
    # TODO (1 line): probability that a Gamma(n, 1) variable is at least score
    pass

In [ ]:
test_p_value(p_value)

<details>
<summary><b>Hint</b></summary>

`stats.gamma.sf(x, a)` is the survival function $\Pr[X \geq x]$ of a Gamma distribution with shape `a`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def p_value(score, n):
    return stats.gamma.sf(score, n)
```
</details>

In [ ]:
for name, tokens in [("plain", plain), ("watermarked", marked)]:
    score, n = watermark_score(tokens, KEY, VOCAB)
    print(f"{name:12s} score {score:6.1f} over {n} tokens, p-value {p_value(score, n):.1e}")

score, n = watermark_score(marked, b"wrong key", VOCAB)
print(f"{'wrong key':12s} score {score:6.1f} over {n} tokens, p-value {p_value(score, n):.1e}")

---

## Experiments

We generate a small corpus of watermarked and plain texts and reuse it below.

In [ ]:
prompts = [
    "The history of cryptography begins with",
    "My favourite recipe for a rainy Sunday is",
    "The committee met on Tuesday to discuss",
    "Scientists studying the deep ocean have found",
    "When I moved to the city, the first thing I noticed was",
    "The best way to learn a new language is",
]
N_TOKENS = 150
marked_corpus = [generate(model, tokenizer, p, watermark_sampler(KEY), N_TOKENS) for p in prompts]
plain_corpus = [generate(model, tokenizer, p, plain_sampler(i), N_TOKENS) for i, p in enumerate(prompts)]


def log10_p(tokens, key=KEY):
    score, n = watermark_score(tokens, key, VOCAB)
    return stats.gamma.logsf(score, n) / np.log(10)

### Experiment 1: Text Length

Every token adds a little evidence. How many tokens does the detector need?

In [ ]:
lengths = np.arange(10, N_TOKENS + 1, 10)
for corpus, color, label in [(marked_corpus, "C1", "watermarked"), (plain_corpus, "C0", "plain")]:
    curves = np.array([[log10_p(tokens[:n]) for n in lengths] for tokens in corpus])
    plt.plot(lengths, curves.T, color=color, alpha=0.3)
    plt.plot(lengths, np.median(curves, axis=0), color=color, linewidth=3, label=label)
plt.axhline(-6, color="gray", linestyle="--", label="p = 1e-6")
plt.xlabel("tokens seen by the detector")
plt.ylabel("log10 p-value")
plt.legend()
plt.show()

### Experiment 2: Entropy

The watermark lives in the choices the model leaves open. Lowering the temperature makes the model more certain of its next token, which removes those choices.

In [ ]:
temperatures = [1.0, 0.8, 0.6, 0.4]
results = [
    [log10_p(generate(model, tokenizer, p, watermark_sampler(KEY), 100, temperature=t)) for p in prompts[:4]]
    for t in temperatures
]
plt.plot(temperatures, results, "o", color="C1", alpha=0.4)
plt.plot(temperatures, np.median(results, axis=1), color="C1", linewidth=3)
plt.xlabel("temperature")
plt.ylabel("log10 p-value after 100 tokens")
plt.show()

### Experiment 3: Editing the Text

An adversary who wants to remove the watermark can edit the text. We simulate this by replacing a fraction of the tokens with random ones. A replaced token breaks its own evidence and that of the `CONTEXT` tokens that follow it.

In [ ]:
def corrupt(tokens, fraction, rng):
    tokens = np.array(tokens)
    replace = rng.random(len(tokens)) < fraction
    tokens[replace] = rng.integers(0, VOCAB, size=replace.sum())
    return tokens.tolist()


rng = np.random.default_rng(0)
fractions = np.linspace(0, 0.5, 11)
curves = np.array([[log10_p(corrupt(tokens, f, rng)) for f in fractions] for tokens in marked_corpus])
plt.plot(fractions, curves.T, color="C1", alpha=0.3)
plt.plot(fractions, np.median(curves, axis=0), color="C1", linewidth=3)
plt.axhline(-6, color="gray", linestyle="--", label="p = 1e-6")
plt.xlabel("fraction of tokens replaced")
plt.ylabel("log10 p-value")
plt.legend()
plt.show()

### Questions

1. Generate twice from the same prompt with the same key. What happens, and why would that be a problem in a product?
2. Why does a larger `CONTEXT` make the watermark easier to remove? What goes wrong if `CONTEXT` is very small?
3. Every single token is a fair sample from $p$. Does that mean an adversary **without** the key can never notice the watermark, even after seeing many responses?

---

## Bonus: From Watermark to Steganography

The watermark transmits one bit: "this text was made with key $k$". To transmit a message, let the pseudorandom numbers depend on the message as well:

$$r = F_k(\text{context}, b)$$

where $b$ is the message bit that is currently being sent. We spend `block_len` tokens on every bit.

The decoder does not know $b$, so it tries both. It computes the detection score of each block once with $b = 0$ and once with $b = 1$. The correct bit gives the higher score.

In [ ]:
def stego_sampler(key, message, block_len):
    def sampler(probs, history, i):
        context = tuple(history[-CONTEXT:]) + (message[i // block_len],)
        return gumbel_sample(probs, prf(key, context, len(probs)))
    return sampler


def text_to_bits(text):
    return [int(b) for char in text.encode() for b in f"{char:08b}"]


def bits_to_text(bits):
    return bytes(int("".join(map(str, bits[i:i + 8])), 2) for i in range(0, len(bits), 8)).decode(errors="replace")

In [ ]:
def decode_message(tokens, key, vocab_size, n_bits, block_len, context_size=CONTEXT):
    message = []
    for j in range(n_bits):
        scores = [0.0, 0.0]
        for i in range(max(j * block_len, context_size), (j + 1) * block_len):
            context = tuple(tokens[i - context_size:i])
            # TODO (2 lines): for bit 0 and bit 1, add -log(1 - r) of the observed token to scores[bit],
            # where r comes from the prf with the bit appended to the context
            pass
        message.append(int(scores[1] > scores[0]))
    return message

In [ ]:
test_decode_message(decode_message, prf)

<details>
<summary><b>Hint</b></summary>

The encoder used `prf(key, context + (bit,), vocab_size)`. The decoder has to use the same input.
</details>

<details>
<summary><b>Solution</b></summary>

```python
def decode_message(tokens, key, vocab_size, n_bits, block_len, context_size=CONTEXT):
    message = []
    for j in range(n_bits):
        scores = [0.0, 0.0]
        for i in range(max(j * block_len, context_size), (j + 1) * block_len):
            context = tuple(tokens[i - context_size:i])
            for bit in (0, 1):
                scores[bit] += -np.log(1 - prf(key, context + (bit,), vocab_size)[tokens[i]])
        message.append(int(scores[1] > scores[0]))
    return message
```
</details>

In [ ]:
secret = text_to_bits("hi")
BLOCK = 20
stego = generate(model, tokenizer, prompts[3], stego_sampler(KEY, secret, BLOCK), len(secret) * BLOCK)

print(tokenizer.decode(stego))
print()
print("with the key:    ", repr(bits_to_text(decode_message(stego, KEY, VOCAB, len(secret), BLOCK))))
print("with a wrong key:", repr(bits_to_text(decode_message(stego, b"wrong key", VOCAB, len(secret), BLOCK))))

### Questions

1. Try smaller values of `BLOCK`. When do decoding errors appear? How does this depend on the prompt and the temperature?
2. A fixed block length wastes tokens on high-entropy text and fails on low-entropy text. The scheme from the lecture instead moves to the next symbol once a score crosses a threshold. Why does it then need the backspace symbol $\leftarrow$?